In [30]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision 
from torchvision.datasets import CIFAR10

In [31]:
# Datasets and dataloader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image =>scale(0,1) => normalize(-1,1)  this the stndrd process
transform = transforms.Compose([
    transforms.ToTensor(), # images ko pytoech tensor me cnvrt krta h + scale bhi
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
])
trainset = CIFAR10(root="\.data",train = True,download = True,transform = transform)
testset = CIFAR10(root="\.data",train = False,download = True,transform = transform)

In [32]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: \.data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [33]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

### Build CNN

In [62]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        #2 layer hogi convolutonal fullyconnected
        self.conv_layers = nn.Sequential(
            # first convolutonal layer
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernal size =2,stride =2

             # second convolutonal layer
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernal size =2,stride =2

             # third convolutonal layer
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernal size =2,stride =2
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256,10) #input me 256 aur op m 10 clss chahiye
        )
    def forward(self,x):
            x = self.conv_layers(x)
            x  = x.view(x.size(0),-1) #flattening
            x = self.fc_layers(x)

            return x

        


In [63]:
model = CNN()

In [64]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [65]:
epochs = 10
for epoch in range(epochs):
    epoch_trainning_loss = 0.0

    for images , labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output , labels)
        loss.backward()
        optimizer.step()
        
        epoch_trainning_loss += loss.item()

    print(f"epoch ={epoch+1}/{epochs} & loss  = {epoch_trainning_loss/len(trainloader)}")


epoch =1/10 & loss  = 1.3724379318449504
epoch =2/10 & loss  = 0.9514102694171164
epoch =3/10 & loss  = 0.7682858572896484
epoch =4/10 & loss  = 0.6376512899346973
epoch =5/10 & loss  = 0.5339702881320053
epoch =6/10 & loss  = 0.4378420581560001
epoch =7/10 & loss  = 0.35456347982863634
epoch =8/10 & loss  = 0.27891149094609347
epoch =9/10 & loss  = 0.21700945685681938
epoch =10/10 & loss  = 0.16911729376124757


In [71]:
# evaluate our cnn
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images,labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy : {correct_labels/total_labels*100}")
    

accuracy : 74.51
